In [3]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_curve, auc

# Suppress common warnings for a cleaner output
warnings.filterwarnings('ignore')

def run_combined_comparative_analysis(
    main_correct_path, 
    main_misclassified_path, 
    wtwt_evaluation_folder, 
    output_dir="model_comparison_plots"
):
    """
    Combines main data with WT-WT data, performs balancing on the entire dataset,
    then splits for training and testing of LR, SVM, and XGBoost models.
    """
    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: Load and Combine All Datasets ---
    print("Loading and combining all data sources...")
    
    # Load main dataset
    df_main_cor = pd.read_csv(main_correct_path)
    df_main_mis = pd.read_csv(main_misclassified_path)
    df_main_cor["label"] = "Correct"
    df_main_mis["label"] = "Misclassified"
    df_main = pd.concat([df_main_cor, df_main_mis])

    # Load WT-WT dataset and apply labels
    df_wtwt = pd.read_csv(os.path.join(wtwt_evaluation_folder, "wtwt_test_processed.csv"))
    corr_idx = np.load(os.path.join(wtwt_evaluation_folder, "correctly_classified_indices.npy"))
    mis_idx = np.load(os.path.join(wtwt_evaluation_folder, "misclassified_indices.npy"))
    df_wtwt["label"] = "Unknown"
    df_wtwt.loc[corr_idx, "label"] = "Correct"
    df_wtwt.loc[mis_idx, "label"] = "Misclassified"
    df_wtwt = df_wtwt[df_wtwt["label"] != "Unknown"]

    # Combine into a single pool
    df_combined = pd.concat([df_main, df_wtwt], ignore_index=True)
    print(f"Total combined examples before balancing: {len(df_combined)}")

    # --- Step 2: Balance the Entire Combined Dataset ---
    print("Balancing the entire combined dataset using stratified undersampling...")
    df_combined_correct = df_combined[df_combined['label'] == 'Correct']
    df_combined_mis = df_combined[df_combined['label'] == 'Misclassified']
    
    mis_stance_counts = df_combined_mis['stance'].value_counts()
    df_combined_correct_sampled = (
        df_combined_correct.groupby('stance')
        .apply(lambda x: x.sample(
            n=min(len(x), mis_stance_counts.get(x.name, 0)),
            random_state=42
        ))
        .reset_index(drop=True)
    )
    
    # The final balanced dataframe for the whole experiment
    df_balanced = pd.concat([df_combined_correct_sampled, df_combined_mis], ignore_index=True)
    df_balanced["label_bin"] = df_balanced["label"].map({"Correct": 1, "Misclassified": 0})
    print(f"Final balanced dataset size: {len(df_balanced)}\n")

    # --- Step 3: Define Models and Features ---
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'SVM': SVC(kernel='rbf', probability=True, random_state=42),
        'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    }

    feature_cols = [c for c in df_balanced.columns if c not in ['target', 'text', 'stance', 'label', 'label_bin'] and pd.api.types.is_numeric_dtype(df_balanced[c])]
    stances = [("All Stances", None), ("AGAINST", 0), ("FAVOR", 1), ("NONE", 2)]

    # --- Step 4: Iterate, Split the Balanced Data, and Evaluate ---
    for stance_name, stance_val in stances:
        print(f"\n{'='*70}\nProcessing Stance: {stance_name}\n{'='*70}")

        df_stance = df_balanced.copy()
        if stance_val is not None:
            df_stance = df_balanced[df_balanced["stance"] == stance_val]

        if df_stance.empty or len(df_stance['label_bin'].unique()) < 2:
            print(f"Skipping {stance_name} due to insufficient data.")
            continue
            
        X = df_stance[feature_cols]
        y = df_stance["label_bin"]

        # Split the already-balanced data into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )
        
        # Preprocessing
        imputer = SimpleImputer(strategy='median')
        X_train_imputed = imputer.fit_transform(X_train)
        X_test_imputed = imputer.transform(X_test)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_imputed)
        X_test_scaled = scaler.transform(X_test_imputed)

        plt.figure(figsize=(8, 7))
        
        for model_name, model in models.items():
            print(f"\n--- {model_name} ---")
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            print(classification_report(y_test, y_pred, target_names=['Misclassified', 'Correct'], digits=4))

            # ROC Curve Logic
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f'{model_name} (AUC = {roc_auc:.2f})')
        
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves for Stance: {stance_name}'); plt.legend(loc="lower right"); plt.grid(True)
        plot_path = os.path.join(output_dir, f"roc_curves_{stance_name.replace(' ', '_')}.png")
        plt.savefig(plot_path, dpi=300); plt.close()
        print(f"\nSaved ROC curve plot to: {plot_path}")

if __name__ == '__main__':
    sns.set_theme(style="whitegrid")
    
    # --- DEFINE YOUR FILE PATHS HERE ---
    MAIN_CORRECT_PATH = '/home/p/parush/style_markers/classifications/whole/correctly_classified_examples_processed.csv'
    MAIN_MISCLASSIFIED_PATH = '/home/p/parush/style_markers/classifications/whole/misclassified_examples_processed.csv'
    WTWT_EVALUATION_FOLDER = 'evaluation_data' # Folder with wtwt_test_processed.csv etc.
    
    run_combined_comparative_analysis(
        main_correct_path=MAIN_CORRECT_PATH,
        main_misclassified_path=MAIN_MISCLASSIFIED_PATH,
        wtwt_evaluation_folder=WTWT_EVALUATION_FOLDER
    )

Loading and combining all data sources...
Total combined examples before balancing: 10622
Balancing the entire combined dataset using stratified undersampling...
Final balanced dataset size: 1600


Processing Stance: All Stances

--- Logistic Regression ---
               precision    recall  f1-score   support

Misclassified     0.5298    0.5563    0.5427       160
      Correct     0.5329    0.5062    0.5192       160

     accuracy                         0.5312       320
    macro avg     0.5313    0.5312    0.5310       320
 weighted avg     0.5313    0.5312    0.5310       320


--- SVM ---
               precision    recall  f1-score   support

Misclassified     0.5450    0.6438    0.5903       160
      Correct     0.5649    0.4625    0.5086       160

     accuracy                         0.5531       320
    macro avg     0.5549    0.5531    0.5494       320
 weighted avg     0.5549    0.5531    0.5494       320


--- XGBoost ---
               precision    recall  f1-score  